# Benchmark DWN × WiSARD

Tabelas do DWN (`torch-dwn`) comparado com a WiSARD treinada por épocas e com a one-shot, nos mesmos conjuntos, binarizações e sementes.

Os resultados do DWN são produzidos pelo script **`experiments/run_dwn.py`** (precisa de GPU NVIDIA; instruções no início do script), que grava uma linha por configuração em `experiments/results_dwn/`:
```
python experiments/run_dwn.py --dry-run     # o que vai rodar
python experiments/run_dwn.py               # tudo; retoma se for interrompido
```
Mesma divisão (`run_grid.split` com as sementes da WiSARD), mesma binarização (`encoders.py`, mesmos bits) e configuração fixa do DWN (a do exemplo do README do `torch-dwn`); só os bits variam. Cada linha tem a acurácia de validação e de teste, o tempo de treino (`secondsTrain`) e o de inferência (`secondsInfer` = validação + teste).

**Escolha**: por semente, o DWN com os bits de maior acurácia de **validação** (empate: menos bits); a WiSARD com as configurações escolhidas pelo `summarize.py` (as mesmas do `benchmark_table.ipynb`).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# the notebook lives in experiments/, but also works when started from the repository root
HERE = Path.cwd() if (Path.cwd() / "summarize.py").exists() else Path.cwd() / "experiments"
sys.path.insert(0, str(HERE))
import summarize as sm

WISARD_RESULTS = HERE / "results"
DWN_RESULTS = HERE / "results_dwn"
ENCODINGS = {"distributive": "Termômetro distributivo", "thermometer": "Termômetro linear"}
MODELS = {"dwn": "DWN", "epochs + bleaching": "WiSARD por épocas", "one-shot + bleaching": "WiSARD one-shot"}


def load_dwn():
    rows = []
    for path in sorted(DWN_RESULTS.glob("*/*/seed*.jsonl")):
        with open(path) as f:
            rows += [json.loads(line) for line in f]
    return [r for r in rows if "error" not in r]


dwn_rows = pd.DataFrame([dict(dataset=r["dataset"], encoding=r["encoding"], seed=r["seed"], bits=r["bits"],
                              val=r["acc"]["val"], test=r["acc"]["test"],
                              train_s=r["secondsTrain"], infer_s=r["secondsInfer"]) for r in load_dwn()],
                        columns=["dataset", "encoding", "seed", "bits", "val", "test", "train_s", "infer_s"])
print("%d configurações do DWN em %d conjuntos/binarizações" % (len(dwn_rows), dwn_rows.groupby(["dataset", "encoding"]).ngroups))

## Todas as configurações do DWN

Uma linha por conjunto × binarização × semente × bits: acurácias e tempos (s).

In [ ]:
dwn_rows.sort_values(["encoding", "dataset", "seed", "bits"]).style.format(
    {"val": "{:.4f}", "test": "{:.4f}", "train_s": "{:.2f}", "infer_s": "{:.3f}"}, decimal=",").hide(axis="index")

## Tabelas: DWN × WiSARD

Acurácia de **teste** em %, média ± desvio-padrão entre as sementes em que o DWN rodou; Δ = DWN − WiSARD por épocas (média das diferenças por semente). Os tempos são do DWN escolhido, média entre as sementes.

In [ ]:
# per dataset/encoding/seed: the bits with the best validation accuracy (ties: fewer bits)
best_dwn = (dwn_rows.sort_values(["val", "bits"], ascending=[False, True])
            .groupby(["dataset", "encoding", "seed"]).head(1).assign(model="dwn"))
wisard = sm.load(str(WISARD_RESULTS))
best_wisard = pd.DataFrame([dict(dataset=dataset, encoding=encoding, seed=seed, model=model, test=test["acc"])
                            for (dataset, encoding), seeds in wisard.items()
                            for seed, rows in seeds.items()
                            for model, (config, test, val) in sm.choose(rows).items()],
                           columns=["dataset", "encoding", "seed", "model", "test"])
# only the seeds where the DWN ran
test_acc = pd.concat([best_dwn, best_wisard.merge(best_dwn[["dataset", "encoding", "seed"]], on=["dataset", "encoding", "seed"])])


def pt(value, decimals=2):
    return ("{:.%df}" % decimals).format(value).replace(".", ",")


def mean_sd(values, scale=100, decimals=2):
    values = scale * np.asarray(values, dtype=float)
    return pt(values.mean(), decimals) + (" ± " + pt(values.std(ddof=1), decimals) if len(values) > 1 else "")


def dwn_table(encoding):
    rows = []
    for dataset, g in test_acc[test_acc["encoding"] == encoding].groupby("dataset"):
        by_model = {m: g[g["model"] == m].sort_values("seed") for m in MODELS}
        dwn = by_model["dwn"]
        row = {"conjunto": dataset, "sementes": dwn["seed"].nunique(), "bits do DWN": ", ".join(str(int(b)) for b in dwn["bits"])}
        for model, label in MODELS.items():
            row[label] = mean_sd(by_model[model]["test"]) if len(by_model[model]) else "—"
        paired = dwn.merge(by_model["epochs + bleaching"], on="seed")
        row["Δ DWN − épocas (p.p.)"] = pt(100 * (paired["test_x"] - paired["test_y"]).mean()) if len(paired) else "—"
        row["DWN: treino (s)"] = mean_sd(dwn["train_s"], scale=1)
        row["DWN: inferência (s)"] = mean_sd(dwn["infer_s"], scale=1, decimals=3)
        rows.append(row)
    return pd.DataFrame(rows)

### Termômetro distributivo

In [ ]:
dwn_table("distributive").style.hide(axis="index")

### Termômetro linear

In [ ]:
dwn_table("thermometer").style.hide(axis="index")